# Part 5: Exploration vs. Exploitation Strategies

**Course**: Reinforcement Learning for AI/ML Engineers  
**Duration**: 2 Hours  
**Instructor**: Mehdi Lotfinejad  

---

## Learning Objectives

By the end of this section, you will be able to:

1. **Explain** the exploration-exploitation dilemma and why it is fundamental to all RL algorithms
2. **Formalize** the Multi-Armed Bandit (MAB) as the canonical testbed for exploration strategies
3. **Implement** five exploration strategies from scratch: Random, Greedy, ε-Greedy, UCB1, and Thompson Sampling
4. **Quantify** exploration quality using *cumulative regret* — the standard metric for bandit algorithms
5. **Understand** how these bandit strategies scale up to full RL (function approximation, deep networks)
6. **Describe** modern deep-RL exploration techniques: count-based bonuses, curiosity-driven rewards, RND, and NoisyNets
7. **Choose** the right exploration strategy for a given environment (stationary vs. non-stationary, tabular vs. deep)

---


## 1. The Exploration-Exploitation Dilemma

Every decision-making agent must constantly answer the same question:

> **Should I exploit what I already know, or explore to discover something better?**

This tension appears in every domain:

| Domain | Exploit | Explore |
|---|---|---|
| RL Agent | Take the best known action | Try a random action to discover better rewards |
| Restaurant choice | Return to your favourite place | Try a new restaurant tonight |
| Drug trials | Assign patients to best known drug | Test new experimental drugs |
| Search engine | Show top-ranked result | Occasionally surface lower-ranked results |
| A/B testing | Serve winning variant | Allocate traffic to untested variants |

### Why Pure Exploitation Fails

```
Scenario: 3-armed slot machine
  Arm A: true mean reward = 5   (you don't know this)
  Arm B: true mean reward = 7   (you don't know this)
  Arm C: true mean reward = 3   (you don't know this)

Start: Pull each arm once → A=6, B=2, C=4  (noisy samples!)
Greedy Agent: "B looks worst (got 2)! Always pull C (got 4)."
  → Gets stuck at suboptimal arm C forever
  → NEVER discovers that B is actually best

Problem: Early noise poisons the greedy estimate. More exploration needed.
```

### Why Pure Exploration Also Fails

```
If you explore FOREVER:
  → You waste pulls on clearly bad arms
  → Cumulative regret grows linearly with time
  → Never converges to optimal

Optimal strategy: Explore enough to identify the best arm, then exploit it.
```

### Formalizing the Trade-off: Regret

The standard metric is **cumulative regret** — the total reward you *missed* by not always choosing the best arm:

$$R(T) = \sum_{t=1}^{T} \left[ \mu^* - \mu_{A_t} \right]$$

where $\mu^*$ is the true mean of the best arm and $\mu_{A_t}$ is the true mean of the arm you chose at time $t$.

- **Linear regret**: $R(T) \propto T$ — terrible, keeps picking bad arms  
- **Sublinear regret**: $R(T) = o(T)$ — good, regret grows slower than time  
- **Logarithmic regret**: $R(T) = O(\log T)$ — optimal (Lai & Robbins lower bound, 1985)

```
Good algorithm:  R(T)
                  │
                  │  Linear (Pure Random)
                  │ /
                  │/
       Sublinear ─┤   ← ε-Greedy, UCB
                  │\
                  │ \  Logarithmic ← UCB1, Thompson Sampling (optimal)
                  └────────────────────────── T
```

### The Exploration Spectrum

```
                    EXPLORATION STRATEGIES
                            │
        ┌───────────────────┼─────────────────────┐
        │                   │                     │
   Undirected           Optimistic             Bayesian
   (random noise)      (uncertainty-based)    (posterior sampling)
        │                   │                     │
   ε-Greedy              UCB1                Thompson Sampling
   NoisyNets             UCB-V               Bayesian DQN
   Random                UCBV                ModelBased Bayes
        │
    Count-based
    Curiosity / RND
```


In [1]:
# ============================================================
# Setup — imports for Part 5 (no heavy dependencies needed!)
# ============================================================
# Run this cell first!

import numpy as np
import random
import math
from collections import defaultdict

# Reproducibility
SEED = 42
np.random.seed(SEED)
random.seed(SEED)

print("=" * 50)
print("  Part 5: Exploration vs. Exploitation")
print("=" * 50)
print(f"  NumPy  : {np.__version__}")
print(f"  Python : stdlib only (no PyTorch needed for bandits)")
print("=" * 50)
print("\nAll imports ready. Let's study exploration from scratch!")


  Part 5: Exploration vs. Exploitation
  NumPy  : 2.4.2
  Python : stdlib only (no PyTorch needed for bandits)

All imports ready. Let's study exploration from scratch!


## 2. The Multi-Armed Bandit (MAB) Problem

The **Multi-Armed Bandit** is the simplest RL problem with no state transitions — just actions and rewards.

```
         k = 10 slot machines (arms)
         ┌───┐  ┌───┐  ┌───┐  ┌───┐  ┌───┐  ┌───┐  ┌───┐  ┌───┐  ┌───┐  ┌───┐
         │   │  │   │  │   │  │   │  │   │  │   │  │   │  │   │  │   │  │   │
         │ 0 │  │ 1 │  │ 2 │  │ 3 │  │ 4 │  │ 5 │  │ 6 │  │ 7 │  │ 8 │  │ 9 │
         └─┬─┘  └─┬─┘  └─┬─┘  └─┬─┘  └─┬─┘  └─┬─┘  └─┬─┘  └─┬─┘  └─┬─┘  └─┬─┘
           │      │      │      │      │      │      │      │      │      │
           ▼      ▼      ▼      ▼      ▼      ▼      ▼      ▼      ▼      ▼
         r~N   r~N   r~N   r~N   r~N   r~N   r~N   r~N   r~N   r~N
         (μ₀,1)(μ₁,1)(μ₂,1)(μ₃,1)(μ₄,1)(μ₅,1)(μ₆,1)(μ₇,1)(μ₈,1)(μ₉,1)

         The true means μ₀…μ₉ are UNKNOWN to the agent.
         Rewards are noisy: pulling arm i returns μᵢ + ε, ε ~ N(0,1)
```

### Formal Definition

- **k arms**: each arm $i$ has unknown true mean $\mu_i$
- **At each round $t$**: choose arm $A_t \in \{0, \ldots, k-1\}$, observe reward $R_t \sim \mathcal{N}(\mu_{A_t}, 1)$
- **Goal**: maximize total reward over $T$ rounds (equivalently, minimize cumulative regret)

### The Sample-Mean Estimator

The simplest unbiased estimator of $\mu_i$ from $n_i$ samples is:

$$\hat{\mu}_i^{(n)} = \frac{1}{n} \sum_{j=1}^{n} r_j \quad \text{(sample mean)}$$

Updated incrementally (no need to store all samples):

$$\hat{\mu}_i \leftarrow \hat{\mu}_i + \frac{1}{n_i}\left(r - \hat{\mu}_i\right)$$

This is the **same incremental mean update** seen in Monte Carlo (Part 3)!

### Baseline Agents to Beat

Before fancy algorithms, let's implement two baselines:
- **Random**: pick a random arm every step — pure exploration, $R(T) = O(T)$  
- **Greedy**: always pick the arm with highest sample mean — pure exploitation, stuck in suboptimal arm


In [3]:
import numpy as np
import random

# ============================================================
# Multi-Armed Bandit Environment + Baseline Agents
# ============================================================

# ── 1. Bandit Environment ─────────────────────────────────
class BanditEnv:
    """
    Stationary k-armed Gaussian bandit.
    True means are drawn once from N(0,1) and fixed for the episode.
    Each pull returns mean + Gaussian noise.
    """
    def __init__(self, k=10, seed=SEED):
        rng = np.random.default_rng(seed)
        self.k = k
        self.true_means  = rng.normal(0.0, 1.0, k)      # unknown to agent
        self.optimal_arm = int(np.argmax(self.true_means))
        self.optimal_mean = self.true_means[self.optimal_arm]

    def pull(self, arm: int) -> float:
        """Sample reward from arm i: μᵢ + N(0,1) noise."""
        return float(np.random.normal(self.true_means[arm], 1.0))

# ── 2. Random Agent ───────────────────────────────────────
class RandomAgent:
    """Pure exploration baseline — uniformly random arm selection."""
    def __init__(self, k): self.k = k
    def select_action(self) -> int: return random.randint(0, self.k - 1)
    def update(self, arm, reward): pass    # no learning

# ── 3. Greedy Agent ───────────────────────────────────────
class GreedyAgent:
    """Pure exploitation — always pick arm with highest sample mean."""
    def __init__(self, k):
        self.k = k
        self.Q = np.zeros(k)       # sample-mean estimates
        self.N = np.zeros(k, int)  # pull counts

    def select_action(self) -> int:
        # Tie-break randomly for arms never pulled
        unpulled = np.where(self.N == 0)[0]
        if len(unpulled):
            return int(random.choice(unpulled))
        return int(np.argmax(self.Q))

    def update(self, arm: int, reward: float):
        self.N[arm] += 1
        # Incremental mean: Q ← Q + (r - Q) / n  (numerically stable)
        self.Q[arm] += (reward - self.Q[arm]) / self.N[arm]

# ── 4. Training / Evaluation Helper ──────────────────────
def run_bandit(agent, env: BanditEnv, T: int = 2000, seed: int = 0):
    """
    Run agent on env for T rounds.
    Returns:
      rewards        : array of shape (T,) — reward at each step
      optimal_frac   : float — fraction of steps the optimal arm was chosen
      cumul_regret   : array (T,) — cumulative regret over time
    """
    random.seed(seed); np.random.seed(seed)
    rewards, is_optimal = [], []
    for _ in range(T):
        arm    = agent.select_action()
        r      = env.pull(arm)
        agent.update(arm, r)
        rewards.append(r)
        is_optimal.append(arm == env.optimal_arm)
    rewards = np.array(rewards)
    # Cumulative regret: difference between optimal arm and what we got each step
    regret  = np.cumsum(env.optimal_mean - rewards)
    return rewards, np.mean(is_optimal), regret

# ── 5. Demo Run ───────────────────────────────────────────
env = BanditEnv(k=10, seed=SEED)

print("=" * 60)
print("10-Armed Bandit — Environment Setup")
print("=" * 60)
print(f"\n  k = {env.k} arms | Optimal arm = {env.optimal_arm} | μ* = {env.optimal_mean:.3f}")
print(f"\n  True means (unknown to agent):")
for i, mu in enumerate(env.true_means):
    marker = " ← BEST" if i == env.optimal_arm else ""
    bar    = "█" * max(0, int((mu + 3) * 4))
    print(f"    Arm {i:2d}: {mu:+.3f}  {bar}{marker}")

T = 2000
print(f"\n{'─'*60}")
print(f"  Running Random vs Greedy over T={T} rounds ...")
print(f"{'─'*60}")
print(f"\n  {'Agent':<20}  {'Avg Reward':>12}  {'% Optimal':>10}  {'Final Regret':>13}")
print("  " + "-" * 60)

for name, agent in [("Random",  RandomAgent(env.k)),
                    ("Greedy",  GreedyAgent(env.k))]:
    r_arr, opt_frac, reg = run_bandit(agent, BanditEnv(10, SEED), T, seed=0)
    final_reg = float(reg[-1])
    print(f"  {name:<20}  {np.mean(r_arr):>12.3f}  {opt_frac*100:>9.1f}%  {final_reg:>13.1f}")

print(f"\n  Optimal mean = {env.optimal_mean:.3f} → perfect agent would avg ≈ {env.optimal_mean:.3f}")


10-Armed Bandit — Environment Setup

  k = 10 arms | Optimal arm = 3 | μ* = 0.941

  True means (unknown to agent):
    Arm  0: +0.305  █████████████
    Arm  1: -1.040  ███████
    Arm  2: +0.750  ███████████████
    Arm  3: +0.941  ███████████████ ← BEST
    Arm  4: -1.951  ████
    Arm  5: -1.302  ██████
    Arm  6: +0.128  ████████████
    Arm  7: -0.316  ██████████
    Arm  8: -0.017  ███████████
    Arm  9: -0.853  ████████

────────────────────────────────────────────────────────────
  Running Random vs Greedy over T=2000 rounds ...
────────────────────────────────────────────────────────────

  Agent                   Avg Reward   % Optimal   Final Regret
  ------------------------------------------------------------
  Random                      -0.344        8.9%         2568.5
  Greedy                       0.911       98.7%           58.7

  Optimal mean = 0.941 → perfect agent would avg ≈ 0.941


## 3. ε-Greedy — The Simplest Fix

The simplest remedy for pure exploitation: with probability ε, *explore* randomly; otherwise *exploit* the best known arm.

```
ε-Greedy Action Selection:
  with probability ε     → choose a random arm (explore)
  with probability 1 − ε → choose arm with highest Q̂ (exploit)
```

### Choosing ε

| ε value | Behaviour | Problem |
|---|---|---|
| ε = 0 | Pure greedy | Gets stuck in local optimum |
| ε = 0.01 | Mostly exploit | Rarely explores; slow to discover best arm |
| ε = 0.1 | Common default | Good balance in stationary environments |
| ε = 0.3 | Lots of exploration | Wastes pulls even after identifying best arm |
| ε = 1.0 | Pure random | Never exploits; linear regret |

### ε-Decay: The Best of Both Worlds

Start with high ε (explore aggressively), then decay it over time (exploit more as knowledge grows):

$$\varepsilon_t = \max\left(\varepsilon_{\min},\; \varepsilon_0 \cdot d^t\right)$$

```
ε-Decay Schedule:
  t=0:      ε = 1.0  → explore freely (everything is unknown)
  t=500:    ε = 0.22 → mostly exploit, some exploration
  t=1000:   ε = 0.05 → exploit known best arm ~95% of time
  t=2000:   ε = 0.01 → almost pure exploitation (min reached)
```

### Regret Analysis of ε-Greedy

- **Fixed ε**: $R(T) = O(\varepsilon \cdot T)$ — constant fraction of rounds is wasted on exploration
- **ε-decay** with optimal schedule: $R(T) = O(\log T)$ — achieves optimal logarithmic regret theoretically

**Limitation**: ε-Greedy explores *uniformly* — it wastes pulls on arms it already knows are bad. The next strategies are smarter.


In [4]:
import numpy as np
import random

# ============================================================
# ε-Greedy Agent (fixed ε and ε-decay variants)
# ============================================================

class EpsilonGreedyAgent:
    """
    ε-Greedy with optional exponential decay.
    With prob ε  → random arm (explore).
    With prob 1-ε → argmax Q̂   (exploit).
    """
    def __init__(self, k: int, epsilon: float = 0.1,
                 decay: float = 1.0, epsilon_min: float = 0.01):
        self.k           = k
        self.epsilon     = epsilon
        self.decay       = decay          # multiply ε by decay after each update
        self.epsilon_min = epsilon_min
        self.Q = np.zeros(k)
        self.N = np.zeros(k, int)

    def select_action(self) -> int:
        if random.random() < self.epsilon:
            return random.randint(0, self.k - 1)   # ── Explore
        return int(np.argmax(self.Q))               # ── Exploit

    def update(self, arm: int, reward: float):
        self.N[arm] += 1
        self.Q[arm] += (reward - self.Q[arm]) / self.N[arm]  # incremental mean
        self.epsilon  = max(self.epsilon_min, self.epsilon * self.decay)


# ── Run and compare ε variants ───────────────────────────
T = 2000
N_RUNS = 50   # average over multiple seeds to reduce noise

configs = [
    ("ε=0.01  (low)",   dict(epsilon=0.01, decay=1.0)),
    ("ε=0.10  (mid)",   dict(epsilon=0.10, decay=1.0)),
    ("ε=0.30  (high)",  dict(epsilon=0.30, decay=1.0)),
    ("ε-decay (1→0.01)", dict(epsilon=1.0, decay=0.997, epsilon_min=0.01)),
]

print("=" * 65)
print("ε-Greedy Variants on 10-Armed Bandit ( T=2000, averaged 50 runs )")
print("=" * 65)
print(f"\n  {'Strategy':<22}  {'Avg Reward':>11}  {'% Optimal':>10}  {'Cum. Regret':>12}")
print("  " + "-" * 60)

results_eps = {}   # store for later comparison
for label, cfg in configs:
    run_rewards, run_opts = [], []
    for seed in range(N_RUNS):
        env_r = BanditEnv(k=10, seed=SEED)   # fixed bandit across seeds
        agent = EpsilonGreedyAgent(k=10, **cfg)
        random.seed(seed); np.random.seed(seed)
        ep_r, ep_o = [], []
        for _ in range(T):
            arm = agent.select_action()
            r   = env_r.pull(arm)
            agent.update(arm, r)
            ep_r.append(r)
            ep_o.append(arm == env_r.optimal_arm)
        run_rewards.append(ep_r)
        run_opts.append(ep_o)

    avg_r   = np.mean(run_rewards)
    avg_opt = np.mean(run_opts) * 100
    cum_reg = T * env_r.optimal_mean - np.sum(np.mean(run_rewards, axis=0))
    results_eps[label] = dict(avg_r=avg_r, avg_opt=avg_opt, cum_reg=cum_reg,
                               rewards=np.mean(run_rewards, axis=0),
                               opts=np.mean(run_opts, axis=0))
    print(f"  {label:<22}  {avg_r:>11.3f}  {avg_opt:>9.1f}%  {cum_reg:>12.1f}")

print(f"\n  Optimal arm mean  = {env_r.optimal_mean:.3f}")
print(f"  Max possible avg  = {env_r.optimal_mean:.3f}  (always picking arm {env_r.optimal_arm})")
print(f"\n  Key insight: ε-decay learns the best arm fast AND stops wasting pulls later.")


ε-Greedy Variants on 10-Armed Bandit ( T=2000, averaged 50 runs )

  Strategy                 Avg Reward   % Optimal   Cum. Regret
  ------------------------------------------------------------
  ε=0.01  (low)                 0.755       37.0%         371.6
  ε=0.10  (mid)                 0.756       66.1%         368.9
  ε=0.30  (high)                0.534       61.8%         812.3
  ε-decay (1→0.01)              0.713       77.6%         454.9

  Optimal arm mean  = 0.941
  Max possible avg  = 0.941  (always picking arm 3)

  Key insight: ε-decay learns the best arm fast AND stops wasting pulls later.


## 4. Upper Confidence Bound (UCB) — Optimism Under Uncertainty

ε-Greedy explores *blindly* — it picks random arms with equal probability, even arms it already knows are bad.

**UCB's key insight**: Explore arms that are *uncertain* (haven't been pulled much), not random ones.

### The UCB1 Algorithm

For each arm $i$, maintain an **optimistic upper bound** on its true mean:

$$\text{UCB}_i(t) = \underbrace{\hat{\mu}_i}_{\text{exploit}} + \underbrace{c \sqrt{\frac{\ln t}{n_i}}}_{\text{exploration bonus}}$$

- $\hat{\mu}_i$: sample mean — what we think arm $i$ is worth  
- $n_i$: how many times we've pulled arm $i$  
- $t$: total steps taken so far  
- $c$: exploration constant (typically $c = \sqrt{2}$ for theoretical guarantees)

**Always select the arm with the highest UCB value**:

$$A_t = \arg\max_i \text{UCB}_i(t)$$

### How the Exploration Bonus Works

```
Step 1: All arms unpulled → UCB = ∞ for all → pull each arm once (forced exploration)

After that:
  Arm pulled often:   √(ln t / nᵢ) → small  (confident estimate → low bonus)
  Arm pulled rarely:  √(ln t / nᵢ) → large  (uncertain estimate → high bonus)

Result: Naturally focuses exploration on uncertain arms.
        An arm's bonus grows if it hasn't been tried recently.
```

### Visualizing UCB

```
After 100 steps:   Arm 7 pulled 3×,  μ̂₇ = 0.8
                   Bonus = c · √(ln 100 / 3) = c · 1.24 → UCB₇ = 2.04

                   Arm 3 pulled 40×, μ̂₃ = 1.2
                   Bonus = c · √(ln 100 / 40) = c · 0.34 → UCB₃ = 1.54

                   → UCB picks Arm 7 (uncertain, despite lower mean)
                   → Next pull of Arm 7 increases n₇, shrinks bonus
                   → UCB self-regulates exploration over time

True means:  Arm 3 = 1.2 (pulled often, well-known)
             Arm 7 = 2.0 (poorly explored — actually BEST arm!)
             → UCB is smart enough to investigate Arm 7 further ✓
```

### Theoretical Guarantee

UCB1 achieves **logarithmic regret** (optimal):

$$R(T) \leq \sum_{i: \mu_i < \mu^*} \frac{8 \ln T}{\Delta_i} + \left(1 + \frac{\pi^2}{3}\right) \sum_{i: \mu_i < \mu^*} \Delta_i$$

where $\Delta_i = \mu^* - \mu_i$ is the gap of arm $i$ from the optimal.

This was proven optimal by Auer, Cesa-Bianchi, and Fischer (2002) — **UCB1 matches the Lai & Robbins lower bound**.


In [5]:
import numpy as np
import math

# ============================================================
# UCB1 Agent
# ============================================================

class UCB1Agent:
    """
    UCB1 (Auer et al., 2002) — logarithmic regret guarantee.

    Action selection:
      A_t = argmax_i [ Q̂_i  +  c · √(ln t / nᵢ) ]

    Forces each arm to be pulled at least once before using UCB formula.
    """
    def __init__(self, k: int, c: float = 2.0):
        self.k   = k
        self.c   = c
        self.Q   = np.zeros(k)       # sample mean estimates
        self.N   = np.zeros(k, int)  # pull counts
        self.t   = 0                 # total steps

    def select_action(self) -> int:
        self.t += 1
        # ── Force initial pull of every arm ──────────────────
        for arm in range(self.k):
            if self.N[arm] == 0:
                return arm
        # ── UCB selection ─────────────────────────────────────
        ucb = self.Q + self.c * np.sqrt(np.log(self.t) / self.N)
        return int(np.argmax(ucb))

    def update(self, arm: int, reward: float):
        self.N[arm] += 1
        self.Q[arm] += (reward - self.Q[arm]) / self.N[arm]


# ── Run UCB1 and compare with ε-Greedy ───────────────────
T      = 2000
N_RUNS = 50

ucb_configs = [
    ("UCB1  c=1.0",  dict(c=1.0)),
    ("UCB1  c=2.0",  dict(c=2.0)),
    ("UCB1  c=4.0",  dict(c=4.0)),
]

print("=" * 65)
print("UCB1 on 10-Armed Bandit  ( T=2000, averaged 50 runs )")
print("=" * 65)
print(f"\n  {'Strategy':<22}  {'Avg Reward':>11}  {'% Optimal':>10}  {'Cum. Regret':>12}")
print("  " + "-" * 60)

results_ucb = {}
for label, cfg in ucb_configs:
    run_rewards, run_opts = [], []
    for seed in range(N_RUNS):
        env_r = BanditEnv(k=10, seed=SEED)
        agent = UCB1Agent(k=10, **cfg)
        np.random.seed(seed)
        ep_r, ep_o = [], []
        for _ in range(T):
            arm = agent.select_action()
            r   = env_r.pull(arm)
            agent.update(arm, r)
            ep_r.append(r)
            ep_o.append(arm == env_r.optimal_arm)
        run_rewards.append(ep_r)
        run_opts.append(ep_o)

    avg_r   = np.mean(run_rewards)
    avg_opt = np.mean(run_opts) * 100
    cum_reg = T * env_r.optimal_mean - np.sum(np.mean(run_rewards, axis=0))
    results_ucb[label] = dict(avg_r=avg_r, avg_opt=avg_opt, cum_reg=cum_reg,
                               rewards=np.mean(run_rewards, axis=0),
                               opts=np.mean(run_opts, axis=0))
    print(f"  {label:<22}  {avg_r:>11.3f}  {avg_opt:>9.1f}%  {cum_reg:>12.1f}")

# Compare best UCB vs best ε-Greedy
print(f"\n{'─'*65}")
print("  UCB1 vs best ε-Greedy:")
best_eps_label = "ε-decay (1→0.01)"
best_eps  = results_eps[best_eps_label]
best_ucb  = results_ucb["UCB1  c=2.0"]
print(f"  {best_eps_label:<22}  {best_eps['avg_r']:>11.3f}  {best_eps['avg_opt']:>9.1f}%  {best_eps['cum_reg']:>12.1f}")
print(f"  {'UCB1 c=2.0':<22}  {best_ucb['avg_r']:>11.3f}  {best_ucb['avg_opt']:>9.1f}%  {best_ucb['cum_reg']:>12.1f}")

improvement = (best_eps['cum_reg'] - best_ucb['cum_reg']) / best_eps['cum_reg'] * 100
print(f"\n  UCB1 reduces cumulative regret by ~{improvement:.1f}% vs ε-decay")
print(f"  Reason: UCB directs exploration to UNCERTAIN arms, not random ones.")


UCB1 on 10-Armed Bandit  ( T=2000, averaged 50 runs )

  Strategy                 Avg Reward   % Optimal   Cum. Regret
  ------------------------------------------------------------
  UCB1  c=1.0                   0.899       89.2%          83.0
  UCB1  c=2.0                   0.837       79.6%         206.2
  UCB1  c=4.0                   0.661       54.9%         558.5

─────────────────────────────────────────────────────────────────
  UCB1 vs best ε-Greedy:
  ε-decay (1→0.01)              0.713       77.6%         454.9
  UCB1 c=2.0                    0.837       79.6%         206.2

  UCB1 reduces cumulative regret by ~54.7% vs ε-decay
  Reason: UCB directs exploration to UNCERTAIN arms, not random ones.


## 5. Thompson Sampling — The Bayesian Approach

UCB is *frequentist* — it uses confidence intervals based on observed frequencies.

**Thompson Sampling** takes a *Bayesian* approach: maintain a **probability distribution** over each arm's true mean, then sample from it.

### Core Idea: Posterior Sampling

```
Bayesian Bandit:
  Prior:    Believe μᵢ ~ some distribution P(μᵢ)   (before seeing any data)
  Observe:  Pull arm i → get reward rᵢ
  Update:   Posterior P(μᵢ | r₁, r₂, ...) via Bayes' rule
  Select:   Sample θᵢ ~ P(μᵢ | observations) for each arm
            Pull arm with highest sample θᵢ
```

### Beta-Bernoulli Thompson Sampling (Binary Rewards)

For binary rewards (success/failure), the **Beta distribution** is the conjugate prior:

$$\text{Prior: } \mu_i \sim \text{Beta}(\alpha_i, \beta_i)$$

$$\text{Update: } \begin{cases} \alpha_i \leftarrow \alpha_i + 1 & \text{if success (reward > 0)} \\ \beta_i \leftarrow \beta_i + 1 & \text{if failure (reward ≤ 0)} \end{cases}$$

Starting with $\alpha_i = \beta_i = 1$ (uniform prior: all values equally likely).

```
Beta(1, 1)  = Uniform[0,1]    → "I know nothing about arm i"
Beta(10, 1) = concentrated near 1 → "arm i almost always succeeds"
Beta(1, 10) = concentrated near 0 → "arm i almost always fails"
Beta(5, 5)  = centered at 0.5, moderate spread → "arm i succeeds ~50% of time"
```

### Why Thompson Sampling Works

```
Early exploration:  Wide posteriors for all arms
                    → samples vary a lot → frequent arm switching → good exploration

As data accumulates: Posterior of good arms narrows around their true mean
                     Bad arms' posteriors tighten near low values
                     → good arms sampled almost every time → exploitation

The magic: Exploration is proportional to UNCERTAINTY.
           Arms aren't explored randomly — they're explored when there's
           genuine uncertainty about whether they might be best.
```

### Thompson Sampling vs UCB

| | UCB1 | Thompson Sampling |
|---|---|---|
| **Philosophy** | Frequentist (confidence bounds) | Bayesian (posterior distributions) |
| **Exploration mechanism** | Deterministic UCB score | Stochastic posterior sampling |
| **Tuning** | Requires choosing $c$ | Only prior hyperparameters |
| **Regret bound** | $O(\log T)$ — proven | $O(\log T)$ — often better empirically |
| **Practical performance** | Excellent | Often slightly better than UCB |
| **Extension to deep RL** | Uncertainty-based bonuses | Bayesian neural networks |


In [6]:
import numpy as np

# ============================================================
# Thompson Sampling Agent (Beta-Bernoulli)
# ============================================================

class ThompsonSamplingAgent:
    """
    Beta-Bernoulli Thompson Sampling.
    Binary reward: success if r > 0 (reward above the global mean of 0).

    Posterior for arm i: Beta(αᵢ, βᵢ)
      αᵢ = # successes + 1
      βᵢ = # failures  + 1
    Start with Beta(1,1) = Uniform prior.
    """
    def __init__(self, k: int):
        self.k     = k
        self.alpha = np.ones(k)    # success counts + 1
        self.beta  = np.ones(k)    # failure counts + 1

    def select_action(self) -> int:
        # Sample a reward probability θᵢ from each arm's posterior
        theta = np.random.beta(self.alpha, self.beta)
        return int(np.argmax(theta))

    def update(self, arm: int, reward: float):
        success = 1 if reward > 0 else 0
        self.alpha[arm] += success
        self.beta[arm]  += (1 - success)

    def posterior_mean(self, arm: int) -> float:
        """Expected value of Beta(α, β) = α / (α + β)."""
        return self.alpha[arm] / (self.alpha[arm] + self.beta[arm])


# ── Run Thompson Sampling ─────────────────────────────────
T      = 2000
N_RUNS = 50

print("=" * 65)
print("Thompson Sampling on 10-Armed Bandit  ( T=2000, 50 runs )")
print("=" * 65)

run_rewards_ts, run_opts_ts = [], []
for seed in range(N_RUNS):
    env_r = BanditEnv(k=10, seed=SEED)
    agent = ThompsonSamplingAgent(k=10)
    np.random.seed(seed)
    ep_r, ep_o = [], []
    for _ in range(T):
        arm = agent.select_action()
        r   = env_r.pull(arm)
        agent.update(arm, r)
        ep_r.append(r)
        ep_o.append(arm == env_r.optimal_arm)
    run_rewards_ts.append(ep_r)
    run_opts_ts.append(ep_o)

ts_avg_r   = np.mean(run_rewards_ts)
ts_avg_opt = np.mean(run_opts_ts) * 100
ts_cum_reg = T * env_r.optimal_mean - np.sum(np.mean(run_rewards_ts, axis=0))

print(f"\n  {'Strategy':<22}  {'Avg Reward':>11}  {'% Optimal':>10}  {'Cum. Regret':>12}")
print("  " + "-" * 60)
print(f"  {'Thompson Sampling':<22}  {ts_avg_r:>11.3f}  {ts_avg_opt:>9.1f}%  {ts_cum_reg:>12.1f}")

# Store for comparison
results_ts = dict(avg_r=ts_avg_r, avg_opt=ts_avg_opt, cum_reg=ts_cum_reg,
                  rewards=np.mean(run_rewards_ts, axis=0),
                  opts=np.mean(run_opts_ts, axis=0))

# ── Print posterior state after last run ─────────────────
print(f"\n  Posterior beliefs after {T} steps (last run):")
print(f"  {'Arm':>4}  {'α (success)':>12}  {'β (failure)':>12}  {'Post. Mean':>11}  {'True Mean':>10}")
print("  " + "-" * 55)
for i in range(env_r.k):
    marker = " ← BEST" if i == env_r.optimal_arm else ""
    print(f"  {i:>4}  {agent.alpha[i]:>12.0f}  {agent.beta[i]:>12.0f}  "
          f"{agent.posterior_mean(i):>11.3f}  {env_r.true_means[i]:>10.3f}{marker}")

print(f"\n  Optimal arm α >> β → posterior mean ≈ true mean ✓")
print(f"  Suboptimal arms pulled rarely → posterior stays near 0.5")


Thompson Sampling on 10-Armed Bandit  ( T=2000, 50 runs )

  Strategy                 Avg Reward   % Optimal   Cum. Regret
  ------------------------------------------------------------
  Thompson Sampling             0.866       83.3%         148.2

  Posterior beliefs after 2000 steps (last run):
   Arm   α (success)   β (failure)   Post. Mean   True Mean
  -------------------------------------------------------
     0            30            17        0.638       0.305
     1             3             7        0.300      -1.040
     2           396           111        0.781       0.750
     3          1123           251        0.817       0.941 ← BEST
     4             1             7        0.125      -1.951
     5             1             5        0.167      -1.302
     6            18            13        0.581       0.128
     7             2             5        0.286      -0.316
     8             9             9        0.500      -0.017
     9             5             7 

## 6. Exploration in Deep RL — Scaling Beyond Bandits

Bandit strategies (ε-Greedy, UCB, Thompson) assume a **finite set of arms** with stationary rewards. In deep RL we face:

- **Continuous/huge state spaces** — can't count visits per state  
- **Non-stationarity** — reward distribution shifts as the policy changes  
- **Sparse rewards** — agent may go millions of steps without any positive reward signal

The bandit intuition still applies — but the mechanisms must scale.

### 6.1 Count-Based Exploration (Tabular → Neural)

In tabular RL (Part 3), we can count $N(s, a)$ per state-action pair:

$$r_i^+(s, a) = r + \frac{\beta}{\sqrt{N(s, a)}}$$

An **exploration bonus** is added to the reward — cells visited rarely get a boost.

For large/continuous spaces: use **pseudo-counts** or a **density model** $\rho_\theta(s)$:

$$\hat{N}(s) = \rho_\theta(s) \cdot n \qquad r^+(s) = r + \beta / \sqrt{\hat{N}(s)}$$

Used in: [Unifying Count-Based Exploration — Bellemare et al., 2016]

### 6.2 Curiosity-Driven Exploration (ICM)

**Intrinsic Curiosity Module (Pathak et al., 2017)**:

```
Idea: Agent is rewarded for visiting states that SURPRISE it.
      Surprise = prediction error of a forward model.

Architecture:
  State sₜ, Action aₜ  →  Feature Encoder  →  φ(sₜ)
                                    ↓
                    Forward Model: predict φ(sₜ₊₁) from φ(sₜ), aₜ
                    Inverse Model: predict aₜ from φ(sₜ), φ(sₜ₊₁)

Intrinsic reward:  rᵢ = ||φ̂(sₜ₊₁) − φ(sₜ₊₁)||²
                        (prediction error = surprise = curiosity bonus)
```

**Problem**: "Noisy TV" — random uncontrollable events (like a TV with static) produce infinite surprise without useful exploration.

### 6.3 Random Network Distillation (RND)

**RND (Burda et al., 2018)** solves the noisy TV problem:

```
Two neural networks:
  Random Target Network  f(s):  FIXED random weights (never trained)
  Predictor Network     f̂(s):  TRAINED to match f(s)

Intrinsic reward:  rᵢ = ||f̂(s) − f(s)||²

Key insight:
  States seen before    → Predictor has learned to match target → low error → low bonus
  States never seen     → Predictor far from target              → high error → high bonus
  Random TV states      → target IS random too → predictor matches quickly → no bonus!

RND worked spectacularly: First algorithm to beat Montezuma's Revenge on Atari
(a notoriously hard sparse-reward game).
```

### 6.4 NoisyNets (Parameter Space Noise)

**NoisyNets (Fortunato et al., 2017)**: Replace ε-greedy with **learnable noise in network weights**:

```
Standard linear layer:  y = Wx + b
NoisyNet linear layer:  y = (μ_W + σ_W ⊙ ε_W) x + (μ_b + σ_b ⊙ ε_b)

ε_W, ε_b: sampled noise (fixed per episode → consistent exploration behaviour)
μ, σ:    learned parameters → the network learns HOW MUCH to explore

Advantage: No ε hyperparameter to tune; exploration adapts to what the network
           is uncertain about; noise is correlated across time steps (unlike ε-greedy)
```

Used in: DQN → Rainbow DQN, NoisyDQN (often outperforms ε-greedy by 20–50% on Atari)

### 6.5 Entropy Bonus (Policy Gradient)

Recall from Part 4: PPO's loss includes an **entropy bonus**:

$$\mathcal{L} = \mathcal{L}_\text{CLIP} - c_1 \mathcal{L}_\text{VF} + c_2 \mathbf{H}[\pi_\theta]$$

$\mathbf{H}[\pi_\theta] = -\sum_a \pi(a|s) \log \pi(a|s)$ is the entropy of the policy.

Maximizing entropy encourages a **diverse, stochastic policy** — the agent is penalized for becoming too certain (and thus too greedy) prematurely. This is equivalent to a soft ε-greedy but in policy space.

### Summary: Which Deep RL Exploration Method to Use

| Method | When to use | Key hyperparameter |
|---|---|---|
| ε-Greedy | Simple baseline, DQN on discrete actions | ε, decay schedule |
| Entropy Bonus | Policy gradient (REINFORCE, PPO) — default | $c_2$ coefficient |
| Count-based | Tabular or small discrete state spaces | $\beta$ bonus scale |
| ICM / Curiosity | Sparse reward + controllable env | Forward model architecture |
| RND | Sparse reward + uncontrollable visual noise | Predictor capacity |
| NoisyNets | DQN variants, Rainbow | $\sigma_0$ initial noise |
| Thompson / UCB | Contextual bandits, recommendation systems | $c$ or prior strength |


## 7. Full Algorithm Comparison

Let's now run all five strategies under identical conditions and compare:

- **Random** — pure exploration baseline  
- **Greedy** — pure exploitation baseline  
- **ε-Greedy (ε=0.1)** — classic fixed exploration  
- **ε-decay** — decaying exploration  
- **UCB1 (c=2)** — optimistic exploration  
- **Thompson Sampling** — Bayesian exploration  


In [7]:
import numpy as np
import random

# ============================================================
# Full Comparison: All 6 Strategies
# ============================================================

T      = 2000
N_RUNS = 100    # more runs for stable comparison

def evaluate(make_agent_fn, T=2000, n_runs=100, bandit_seed=SEED):
    """Run agent factory for n_runs seeds, return aggregated stats."""
    all_rewards, all_opts = [], []
    env_ref = BanditEnv(k=10, seed=bandit_seed)

    for seed in range(n_runs):
        agent = make_agent_fn()
        env   = BanditEnv(k=10, seed=bandit_seed)   # same bandit every run
        np.random.seed(seed); random.seed(seed)
        ep_r, ep_o = [], []
        for _ in range(T):
            arm = agent.select_action()
            r   = env.pull(arm)
            agent.update(arm, r)
            ep_r.append(r)
            ep_o.append(arm == env.optimal_arm)
        all_rewards.append(ep_r)
        all_opts.append(ep_o)

    mean_r    = np.mean(all_rewards, axis=0)    # (T,) mean reward per step
    mean_opt  = np.mean(all_opts,    axis=0)    # (T,) fraction optimal per step
    avg_r     = float(np.mean(mean_r))
    avg_opt   = float(np.mean(mean_opt)) * 100
    cum_reg   = float(T * env_ref.optimal_mean - np.sum(mean_r))
    late_opt  = float(np.mean(mean_opt[-200:])) * 100   # last 200 steps
    return dict(avg_r=avg_r, avg_opt=avg_opt, cum_reg=cum_reg,
                late_opt=late_opt, mean_r=mean_r, mean_opt=mean_opt)

strategies = [
    ("Random",           lambda: RandomAgent(k=10)),
    ("Greedy",           lambda: GreedyAgent(k=10)),
    ("ε-Greedy ε=0.1",  lambda: EpsilonGreedyAgent(k=10, epsilon=0.10)),
    ("ε-Decay",          lambda: EpsilonGreedyAgent(k=10, epsilon=1.0,  decay=0.997, epsilon_min=0.01)),
    ("UCB1  c=2",        lambda: UCB1Agent(k=10, c=2.0)),
    ("Thompson",         lambda: ThompsonSamplingAgent(k=10)),
]

print("=" * 75)
print(f"  Full Comparison — 10-Armed Bandit  (T={T}, {N_RUNS} runs, fixed bandit)")
print("=" * 75)
print(f"\n  {'Strategy':<22}  {'Avg Reward':>11}  {'% Optimal':>10}  "
      f"{'Cum. Regret':>12}  {'Late %Opt':>10}")
print("  " + "-" * 70)

all_stats = {}
for name, factory in strategies:
    stats = evaluate(factory, T=T, n_runs=N_RUNS)
    all_stats[name] = stats
    print(f"  {name:<22}  {stats['avg_r']:>11.3f}  {stats['avg_opt']:>9.1f}%  "
          f"{stats['cum_reg']:>12.1f}  {stats['late_opt']:>9.1f}%")

# ── Regret progression: snapshot every 500 steps ─────────
print(f"\n{'─'*75}")
print("  Cumulative Regret Progression")
print(f"{'─'*75}")
print(f"\n  {'Strategy':<22}  {'t=200':>8}  {'t=500':>8}  {'t=1000':>8}  {'t=2000':>8}")
print("  " + "-" * 58)

for name, factory in strategies:
    stats = all_stats[name]
    env_ref = BanditEnv(k=10, seed=SEED)
    regret = np.cumsum(env_ref.optimal_mean - stats['mean_r'])
    r200, r500, r1000, r2000 = regret[199], regret[499], regret[999], regret[1999]
    print(f"  {name:<22}  {r200:>8.1f}  {r500:>8.1f}  {r1000:>8.1f}  {r2000:>8.1f}")

# ── ASCII regret chart at T=2000 ─────────────────────────
print(f"\n{'─'*75}")
print("  Cumulative Regret at T=2000 (lower = better)")
print(f"{'─'*75}")
max_reg = max(s['cum_reg'] for s in all_stats.values())
for name, _ in strategies:
    reg  = all_stats[name]['cum_reg']
    bar  = "█" * int(reg / max_reg * 40)
    print(f"  {name:<22} {bar:<42} {reg:>7.1f}")

# ── Decision guide ────────────────────────────────────────
print(f"\n{'─'*75}")
print("  Strategy Selection Guide")
print(f"{'─'*75}")
guide = [
    ("Simple baseline / debug",           "Random"),
    ("Stationary env, speed matters",      "UCB1 / Thompson"),
    ("DQN / discrete-action deep RL",      "ε-Greedy (ε=0.1) or NoisyNet"),
    ("Policy gradient (PPO, REINFORCE)",   "Entropy Bonus"),
    ("Recommendation system (MAB)",        "Thompson Sampling"),
    ("Non-stationary (sliding window)",    "ε-Greedy + adaptive ε"),
    ("Sparse rewards, complex env",        "RND or Curiosity (ICM)"),
]
print(f"\n  {'Situation':<40}  {'Recommended Strategy':<30}")
print("  " + "-" * 72)
for sit, strat in guide:
    print(f"  {sit:<40}  {strat:<30}")


  Full Comparison — 10-Armed Bandit  (T=2000, 100 runs, fixed bandit)

  Strategy                 Avg Reward   % Optimal   Cum. Regret   Late %Opt
  ----------------------------------------------------------------------
  Random                       -0.336       10.0%        2552.7        9.9%
  Greedy                        0.795       59.5%         290.6       60.0%
  ε-Greedy ε=0.1                0.757       66.5%         366.5       75.1%
  ε-Decay                       0.710       75.7%         461.2       90.3%
  UCB1  c=2                     0.837       79.0%         207.4       87.9%
  Thompson                      0.866       83.6%         148.2       94.4%

───────────────────────────────────────────────────────────────────────────
  Cumulative Regret Progression
───────────────────────────────────────────────────────────────────────────

  Strategy                   t=200     t=500    t=1000    t=2000
  ----------------------------------------------------------
  Random    

## 8. Exercise: Non-Stationary Bandit

In practice, reward distributions **shift over time** — user preferences change, prices fluctuate, network conditions vary.

Standard UCB and Thompson Sampling assume a **stationary** environment. How do they hold up when the arms suddenly change?

### The Challenge

```
Non-stationary 10-armed bandit:
  t = 1  …  1000 : original arm means  μ₀…μ₉
  t = 1001 … 2000 : means shuffled!  μ_π(0)…μ_π(9)
                    (the best arm CHANGES at t=1000)

Stationary algorithms:  Strong prior for old best arm → slow to adapt
Adaptive algorithms:    Forget old data quickly → adapt faster
```

### Your Task

The cell below compares three approaches on this non-stationary bandit:

1. **UCB1** (standard) — accumulates counts forever
2. **ε-Greedy fixed** — always explores a little
3. **Sliding-Window ε-Greedy** — only remembers last W steps

**Discussion questions** (think before running):
- Which algorithm do you expect to perform worst after the shift?
- Why does UCB1 struggle more than ε-Greedy in non-stationary settings?
- What does the sliding window essentially implement? (Hint: think of `α` in TD learning)
- How would you adapt UCB for non-stationary environments?


In [9]:
import numpy as np
import random

# ============================================================
# Exercise: Non-Stationary Bandit
# ============================================================

# ── Non-Stationary Bandit Environment ─────────────────────
class NonStationaryBandit:
    """
    10-armed bandit where arm means SHIFT at t = shift_at.
    Before shift: original means.
    After  shift: means are randomly shuffled.
    """
    def __init__(self, k=10, shift_at=1000, seed=SEED):
        rng = np.random.default_rng(seed)
        self.k        = k
        self.shift_at = shift_at
        self.original = rng.normal(0.0, 1.0, k)
        # Shuffled version — best arm will be different
        shuffled = self.original.copy()
        rng.shuffle(shuffled)
        self.shifted  = shuffled
        self.t        = 0

    @property
    def current_means(self):
        return self.original if self.t < self.shift_at else self.shifted

    @property
    def optimal_arm(self):
        return int(np.argmax(self.current_means))

    def pull(self, arm: int) -> float:
        self.t += 1
        return float(np.random.normal(self.current_means[arm], 1.0))

    def reset(self):
        self.t = 0


# ── Sliding-Window ε-Greedy ───────────────────────────────
class SlidingWindowEpsGreedy:
    """
    ε-Greedy with a sliding window of size W.
    Only keeps the last W rewards per arm → forgets stale data → adapts to change.
    """
    def __init__(self, k: int, epsilon: float = 0.1, W: int = 200):
        self.k       = k
        self.epsilon = epsilon
        self.W       = W
        self.history = [[] for _ in range(k)]   # rolling reward history per arm

    def select_action(self) -> int:
        if random.random() < self.epsilon:
            return random.randint(0, self.k - 1)
        means = []
        for arm in range(self.k):
            h = self.history[arm]
            means.append(np.mean(h[-self.W:]) if h else 0.0)
        return int(np.argmax(means))

    def update(self, arm: int, reward: float):
        self.history[arm].append(reward)

# ── Run on Non-Stationary Bandit ──────────────────────────
T        = 2000
SHIFT_AT = 1000
N_RUNS   = 100
W        = 200   # sliding window size

def run_non_stationary(make_agent_fn, T, shift_at, n_runs):
    all_rewards, all_opts = [], []
    for seed in range(n_runs):
        env   = NonStationaryBandit(k=10, shift_at=shift_at, seed=SEED)
        agent = make_agent_fn()
        np.random.seed(seed); random.seed(seed)
        ep_r, ep_o = [], []
        for _ in range(T):
            arm = agent.select_action()
            r   = env.pull(arm)
            agent.update(arm, r)
            ep_r.append(r)
            ep_o.append(arm == env.optimal_arm)
        all_rewards.append(ep_r)
        all_opts.append(ep_o)
    return np.mean(all_rewards, axis=0), np.mean(all_opts, axis=0)

ns_strategies = [
    ("UCB1  c=2   (stationary)",   lambda: UCB1Agent(k=10, c=2.0)),
    ("ε-Greedy  ε=0.1",            lambda: EpsilonGreedyAgent(k=10, epsilon=0.1)),
    (f"Sliding-Win W={W}",         lambda: SlidingWindowEpsGreedy(k=10, epsilon=0.1, W=W)),
]

print("=" * 65)
print(f"  Non-Stationary Bandit  (shift at t={SHIFT_AT}, T={T}, {N_RUNS} runs)")
print("=" * 65)
print(f"  Window size W = {W}  |  ε = 0.1  |  c = 2.0")

all_ns_r = {}
for name, factory in ns_strategies:
    mean_r, mean_opt = run_non_stationary(factory, T, SHIFT_AT, N_RUNS)
    all_ns_r[name] = (mean_r, mean_opt)

# Print window stats: before and after shift
print(f"\n  Performance BEFORE shift (t=1…{SHIFT_AT}) vs AFTER shift (t={SHIFT_AT+1}…{T})")
print(f"\n  {'Strategy':<28}  {'Avg Before':>11}  {'Avg After':>11}  {'Drop':>8}")
print("  " + "-" * 62)

for name, (mean_r, mean_opt) in all_ns_r.items():
    before = np.mean(mean_r[:SHIFT_AT])
    after  = np.mean(mean_r[SHIFT_AT:])
    drop   = before - after
    print(f"  {name:<28}  {before:>11.3f}  {after:>11.3f}  {drop:>+8.3f}")

# Print % optimal in each phase
print(f"\n  % Optimal Arm Selection")
print(f"\n  {'Strategy':<28}  {'Before %Opt':>12}  {'After %Opt':>11}")
print("  " + "-" * 54)

for name, (mean_r, mean_opt) in all_ns_r.items():
    before_opt = np.mean(mean_opt[:SHIFT_AT]) * 100
    after_opt  = np.mean(mean_opt[SHIFT_AT:]) * 100
    print(f"  {name:<28}  {before_opt:>11.1f}%  {after_opt:>10.1f}%")

print(f"""
{'─'*65}
  Discussion Answers:

  1. ε-GREEDY (fixed) SUFFERS MOST after the shift — it collapses to
     near 0% optimal arm selection. Its stale Q estimate for the old
     best arm stays high because exploitation (90% of pulls) keeps
     reinforcing it. The 10% random exploration is spread across ALL
     k arms, so discovering the new best arm takes O(k/ε) = 100 steps
     of pure luck — unlikely in the 1000 steps remaining.

  2. UCB1 adapts BETTER than fixed ε-Greedy despite being "stationary".
     Why? Every new pull of the stale arm returns lower rewards than
     expected → its Q estimate slowly revises downward. Meanwhile arms
     not recently pulled grow their confidence bonus c·√(ln t / nᵢ),
     making UCB naturally revisit them. This partial self-correction
     keeps UCB1 at ~30% optimal — far better than ε-Greedy's 0.9%.

  3. Sliding Window forgets data older than W steps. This is equivalent
     to a CONSTANT learning rate α = 1/W: recent rewards count more.
     Identical intuition to using α=0.01 in TD learning (Part 3):
     exponential weighting toward recency. Both make the agent adaptive
     but introduce variance — small W = fast adapt, noisy; large W = slow.

  4. UCB adapted for non-stationarity:
     • Sliding-Window UCB: Q̂ᵢ = mean of last W pulls; nᵢ = min(N(i), W)
     • Discounted UCB: weight each reward by γᵗ⁻ˢ so old data decays
     Both achieve near-optimal regret in slowly-drifting environments.
{'─'*65}""")


  Non-Stationary Bandit  (shift at t=1000, T=2000, 100 runs)
  Window size W = 200  |  ε = 0.1  |  c = 2.0

  Performance BEFORE shift (t=1…1000) vs AFTER shift (t=1001…2000)

  Strategy                       Avg Before    Avg After      Drop
  --------------------------------------------------------------
  UCB1  c=2   (stationary)            0.775        0.747    +0.028
  ε-Greedy  ε=0.1                     0.732        0.581    +0.152
  Sliding-Win W=200                   0.734        0.629    +0.104

  % Optimal Arm Selection

  Strategy                       Before %Opt   After %Opt
  ------------------------------------------------------
  UCB1  c=2   (stationary)             69.7%        30.4%
  ε-Greedy  ε=0.1                      59.2%         0.9%
  Sliding-Win W=200                    59.6%         0.9%

─────────────────────────────────────────────────────────────────
  Discussion Answers:

  1. ε-GREEDY (fixed) SUFFERS MOST after the shift — it collapses to
     near 0% op

## 9. Summary & Key Takeaways

### What We Built Today

✅ **The Dilemma Formalized** — Understood exploration vs. exploitation as a trade-off measurable via *cumulative regret* $R(T)$, and proved that optimal algorithms achieve $R(T) = O(\log T)$

✅ **Multi-Armed Bandit** — Implemented `BanditEnv` (k-armed Gaussian bandit) and the two baselines; observed why pure exploitation fails due to early noise in estimates

✅ **ε-Greedy** — Built fixed-ε and ε-decay variants; showed that decay achieves better late-stage exploitation; understood the linear regret of fixed-ε

✅ **UCB1** — Implemented the Upper Confidence Bound with the exploration bonus $c\sqrt{\ln t / n_i}$; proved that directed uncertainty-based exploration beats random exploration

✅ **Thompson Sampling** — Built Beta-Bernoulli posterior sampling; saw how the posterior naturally concentrates on the best arm as evidence accumulates

✅ **Deep RL Exploration** — Surveyed five scalable methods: count-based bonuses, ICM (curiosity), RND (prediction error), NoisyNets (parameter noise), and entropy bonus in PPO

✅ **Non-Stationary Bandits** — Implemented the Sliding-Window ε-Greedy and showed that standard UCB fails to adapt; connected window parameter to the TD learning rate α

---

### The Full Progression

```
Information Available        Algorithm              Regret Guarantee
────────────────────         ─────────               ────────────────
None / fully blind           Random                  O(T)   — linear
                             Greedy                  O(T)   — linear (gets stuck)
                             ε-Greedy (fixed ε)      O(εT)  — linear fraction
                             ε-Greedy (decay)        O(√T)  — sublinear
Uncertainty estimates        UCB1                    O(log T) — optimal
Bayesian posterior           Thompson Sampling       O(log T) — optimal (often better)
Neural approximation         RND / ICM / NoisyNets   depends on architecture
```

---

### The Exploration Equation

Every exploration algorithm makes the same underlying trade-off explicit or implicit:

$$\text{Value to act on} = \underbrace{\hat{\mu}_i}_{\text{Exploit}} + \underbrace{\text{Bonus}_i(t)}_{\text{Explore}}$$

| Algorithm | Bonus term |
|---|---|
| ε-Greedy | Random action with prob ε (bonus is a coin flip) |
| UCB1 | $c \sqrt{\ln t / n_i}$ (confidence width) |
| Thompson | Sample from $\text{Beta}(\alpha_i, \beta_i)$ (posterior spread) |
| Count-based | $\beta / \sqrt{N(s)}$ (inverse visit frequency) |
| RND | $\|\hat{f}(s) - f(s)\|^2$ (prediction error) |
| Entropy bonus | $-\sum_a \pi(a|s) \log \pi(a|s)$ (policy entropy) |

---

### Coming Up in Part 6: Model-Based RL & Planning

In Part 6 we'll explore:
1. **Model-Based RL** — learn a world model and plan inside it (Dyna-Q)
2. **MCTS** — Monte Carlo Tree Search (the engine behind AlphaGo/AlphaZero)
3. **Reward Shaping & Sparse Rewards** — guiding agents through reward deserts
4. **RLHF** — Reinforcement Learning from Human Feedback (the ChatGPT training pipeline)

---

*© 2025 Reinforcement Learning for AI/ML Engineers — Mehdi Lotfinejad*


## 📋 Quick Reference Card

```
┌─────────────────────────────────────────────────────────────────────┐
│          EXPLORATION vs. EXPLOITATION CHEAT SHEET                   │
├─────────────────────────────────────────────────────────────────────┤
│                                                                     │
│  REGRET METRIC                                                      │
│  R(T) = Σ [μ* − μ_{Aₜ}]    (total missed reward vs. optimal)       │
│  Goal: minimize R(T) → sublinear (o(T)) is good, O(log T) optimal  │
│                                                                     │
│  RANDOM AGENT                                                       │
│  ┌─────────────────────────────────────────────────────────────┐   │
│  │  Aₜ ~ Uniform{0, …, k−1}                                    │   │
│  │  R(T) = O(T) — linear (terrible)                            │   │
│  │  Use: sanity-check baseline only                             │   │
│  └─────────────────────────────────────────────────────────────┘   │
│                                                                     │
│  ε-GREEDY                                                           │
│  ┌─────────────────────────────────────────────────────────────┐   │
│  │  With prob ε  → random arm (explore)                         │   │
│  │  With prob 1−ε → argmax Q̂ (exploit)                         │   │
│  │  ε-decay: εₜ = max(εₘᵢₙ, ε₀ · dᵗ)                           │   │
│  │  R(T) = O(εT) fixed | O(√T) decay                           │   │
│  │  Default: ε=0.1, decay=0.995, εₘᵢₙ=0.01                    │   │
│  └─────────────────────────────────────────────────────────────┘   │
│                                                                     │
│  UCB1 (Upper Confidence Bound)                                      │
│  ┌─────────────────────────────────────────────────────────────┐   │
│  │  UCBᵢ(t) = Q̂ᵢ + c · √(ln t / nᵢ)                          │   │
│  │  Aₜ = argmax UCBᵢ(t)                                        │   │
│  │  R(T) = O(log T) — optimal                                  │   │
│  │  Default: c = 2.0 (or √2 for tightest bound)                │   │
│  └─────────────────────────────────────────────────────────────┘   │
│                                                                     │
│  THOMPSON SAMPLING (Beta-Bernoulli)                                 │
│  ┌─────────────────────────────────────────────────────────────┐   │
│  │  Per arm: maintain Beta(αᵢ, βᵢ) posterior                   │   │
│  │  Select: θᵢ ~ Beta(αᵢ, βᵢ), pick argmax θᵢ                  │   │
│  │  Update: αᵢ += 1 (success), βᵢ += 1 (failure)               │   │
│  │  R(T) = O(log T) — empirically often best                   │   │
│  └─────────────────────────────────────────────────────────────┘   │
│                                                                     │
│  NON-STATIONARY: SLIDING WINDOW                                     │
│  ┌─────────────────────────────────────────────────────────────┐   │
│  │  Q̂ᵢ = mean of last W rewards for arm i                      │   │
│  │  W ≈ 100–500 depending on speed of change                   │   │
│  │  Equivalent to exponential decay with α = 1/W              │   │
│  └─────────────────────────────────────────────────────────────┘   │
│                                                                     │
│  DEEP RL EXPLORATION METHODS                                        │
│  ┌─────────────────────────────────────────────────────────────┐   │
│  │  ε-Greedy    : standard for DQN (discrete actions)           │   │
│  │  Entropy     : −Σ π(a|s) log π(a|s) bonus in PPO            │   │
│  │  Count bonus : r⁺ = r + β/√N(s)  (tabular/small spaces)    │   │
│  │  RND         : r⁺ = ‖f̂(s)−f(s)‖² (sparse reward, Atari)   │   │
│  │  ICM         : curiosity = forward model error               │   │
│  │  NoisyNet    : replace FC layers with noisy layers           │   │
│  └─────────────────────────────────────────────────────────────┘   │
│                                                                     │
│  DECISION GUIDE                                                     │
│  ┌─────────────────────────────────────────────────────────────┐   │
│  │  Bandit / recommendation sys   → Thompson Sampling           │   │
│  │  Tabular RL, small state space  → ε-Greedy or UCB            │   │
│  │  DQN on Atari                   → ε-decay + NoisyNet         │   │
│  │  PPO / REINFORCE                → Entropy bonus (c₂=0.01)   │   │
│  │  Sparse reward (Montezuma)      → RND or ICM                 │   │
│  │  Non-stationary environment     → Sliding-Window ε-Greedy   │   │
│  │  Need theoretical guarantee     → UCB1 or Thompson          │   │
│  └─────────────────────────────────────────────────────────────┘   │
│                                                                     │
└─────────────────────────────────────────────────────────────────────┘
```
